In [ ]:

Chatbot Expert AI Act — Routage deterministe + memoire conditionnelle
3 modes automatiques decides par du code Python :

DIRECT : regex article/considerant → texte mot a mot (0 LLM)
RAG : FAISS pertinent → LLM redige (1 LLM)
WEB : rien dans FAISS → DuckDuckGo + LLM (1 LLM)
2 environnements supportes :

Local : Ollama (Qwen 2.5 3B) — detection automatique
Cloud (Streamlit Cloud) : HuggingFace Inference API (Mistral 7B Instruct)
Cellule 1 — Installation des dépendances
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"])
0
Cellule 2 — Imports
import os
import re
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.documents import Document
from langchain_community.tools import DuckDuckGoSearchRun

print("Imports OK")
Cellule 3 — Configuration
Modifiez les chemins si nécessaire.

# --- CONFIGURATION ---
MD_FILE          = Path("C:\\Users\\Bernard\\Downloads\\RAG_project\\L-202401689FR.000101.fmx.xml.md")
INDEX_DIR        = Path("faiss_index")
MODEL_NAME       = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
TOP_K            = 8
SCORE_THRESHOLD  = 0.35

# Modeles LLM
OLLAMA_MODEL     = "qwen2.5:3b"                        # Local (Ollama)
HF_MODEL         = "mistralai/Mistral-7B-Instruct-v0.3" # Cloud (HuggingFace)

print(f"Fichier source : {MD_FILE} ({'existe' if MD_FILE.exists() else 'INTROUVABLE'})")
print(f"Index FAISS    : {INDEX_DIR} ({'existe' if INDEX_DIR.exists() else 'a construire'})")
Cellule 4 — Fonctions de parsing et chunking
Le texte de l'AI Act est un règlement européen avec une structure très hiérarchisée :

Considérants (1) à (180) : motivations du texte, format |(N)|texte|
13 Chapitres > Sections > 113 Articles numérotés
La stratégie de chunking exploite cette structure : 1 chunk = 1 considérant ou 1 article (découpé par paragraphe si trop long), avec un préfixe hiérarchique et des métadonnées riches.

def clean_text(text: str) -> str:
    """
    Nettoie les artefacts de la conversion Markdown depuis EUR-Lex :
    - |---|---| : séparateurs de tableau Markdown
    - |a)|texte| : listes à puces au format tableau
    - [texte](url) : liens Markdown
    - Sauts de ligne multiples
    """
    text = re.sub(r"\|---\|---\|", "", text) # Supprimer les séparateurs de tableau
    text = re.sub(r"\|([a-z]\))\|(.+?)\|", r"\1 \2", text) # Convertir les listes à puces
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text) # Supprimer les liens Markdown
    text = re.sub(r"\n{3,}", "\n\n", text) # Réduire les sauts de ligne multiples à deux
    return text.strip() # Supprimer les espaces en début et fin de texte


def parse_ai_act(filepath: Path = MD_FILE) -> list[dict]:
    """
    Parse le Markdown de l'AI Act et retourne une liste de chunks.
    
    Chaque chunk = {
        "content": str,       # Texte du chunk avec préfixe hiérarchique
        "metadata": {         # Métadonnées pour le filtrage et l'affichage
            "type":           # "considerant" ou "article"
            "chapter":        # Numéro du chapitre (ex: "III")
            "chapter_title":  # Titre du chapitre
            "section":        # Numéro de section
            "article":        # Numéro d'article
            "title":          # Titre de l'article
            "paragraph":      # Numéro de paragraphe (si article découpé)
        }
    }
    """
    text = Path(filepath).read_text(encoding="utf-8")

    # Normaliser les espaces insécables (\xa0) → espaces normaux
    # Le document EUR-Lex utilise \xa0 dans "CHAPITRE I", "Article 5", etc.
    text = text.replace("\xa0", " ")

    # Supprimer le header YAML, le titre H1 et le bloc Excerpt
    text = re.sub(r"^---.*?---", "", text, count=1, flags=re.DOTALL)
    text = re.sub(r"^#\s+.*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^>\s+.*$", "", text, flags=re.MULTILINE)

    lines = text.split("\n")
    chunks = []

    # Variables de contexte hiérarchique (mises à jour au fil du parsing)
    current_chapter = ""
    current_chapter_title = ""
    current_section = ""
    current_section_title = ""

    # ========================================
    # PHASE 1 : Trouver la frontière considérants / articles
    # ========================================
    chapitre_start = None
    for idx, line in enumerate(lines):
        if line.strip() == "CHAPITRE I":
            chapitre_start = idx
            break

    i = 0  # Pointeur de ligne courant

    # ========================================
    # PHASE 2 : Parser les considérants (avant CHAPITRE I)
    # Format : |(N)|texte du considérant|
    # ========================================
    if chapitre_start:
        for idx in range(chapitre_start):
            m = re.match(r"^\|(\(\d+\))\|(.+)\|$", lines[idx])
            if not m:
                continue
            num, body = m.group(1), m.group(2)
            cleaned = clean_text(body)
            if len(cleaned) < 20:
                continue
            chunks.append({
                "content": f"Considérant {num} : {cleaned}",
                "metadata": {
                    "type": "considerant",
                    "numero": num,
                    "chapter": "",
                    "chapter_title": "",
                    "section": "",
                    "section_title": "",
                    "article": "",
                    "title": f"Considérant {num}",
                    "paragraph": "",
                },
            })
        i = chapitre_start

    # ========================================
    # PHASE 3 : Parser les articles (après CHAPITRE I)
    # On détecte 3 types de marqueurs : CHAPITRE, SECTION, Article
    # ========================================
    while i < len(lines):
        line = lines[i].strip()

        # --- Détecter un CHAPITRE (met à jour le contexte) ---
        match_chap = re.match(r"^CHAPITRE\s+([IVXLC]+)$", line)
        if match_chap:
            current_chapter = match_chap.group(1)
            current_section = ""
            current_section_title = ""
            # Le titre est sur la prochaine ligne non vide
            j = i + 1
            while j < len(lines) and lines[j].strip() == "":
                j += 1
            if j < len(lines):
                current_chapter_title = lines[j].strip()
            i = j + 1
            continue

        # --- Détecter une SECTION (met à jour le contexte) ---
        match_sec = re.match(r"^SECTION\s+(\d+)$", line)
        if match_sec:
            current_section = match_sec.group(1)
            j = i + 1
            while j < len(lines) and lines[j].strip() == "":
                j += 1
            if j < len(lines):
                current_section_title = lines[j].strip()
            i = j + 1
            continue

        # --- Détecter un Article ---
        match_art = re.match(r"^Article\s+(premier|\d+)$", line)
        if match_art:
            art_num = match_art.group(1)
            if art_num == "premier":
                art_num = "1"

            # Titre = prochaine ligne non vide
            j = i + 1
            while j < len(lines) and lines[j].strip() == "":
                j += 1
            art_title = lines[j].strip() if j < len(lines) else ""
            j += 1

            # Collecter le contenu jusqu'au prochain marqueur
            content_lines = []
            while j < len(lines):
                next_line = lines[j].strip()
                if re.match(r"^(Article\s+(premier|\d+)|CHAPITRE\s+[IVXLC]+|SECTION\s+\d+)$", next_line):
                    break
                content_lines.append(lines[j])
                j += 1

            raw_content = "\n".join(content_lines)
            cleaned = clean_text(raw_content)

            if len(cleaned) < 10:
                i = j
                continue

            # Construire le préfixe hiérarchique
            prefix_parts = []
            if current_chapter:
                prefix_parts.append(f"Chapitre {current_chapter} - {current_chapter_title}")
            if current_section:
                prefix_parts.append(f"Section {current_section} - {current_section_title}")
            prefix_parts.append(f"Article {art_num} : {art_title}")
            prefix = " > ".join(prefix_parts)

            full_content = f"{prefix}\n\n{cleaned}"

            # Si trop long → découper par paragraphe numéroté (1.  , 2.  , ...)
            if len(full_content) > 2000:
                paragraphs = re.split(r"\n(?=\d+\.\s{2,})", cleaned)
                for p_idx, para in enumerate(paragraphs):
                    para = para.strip()
                    if len(para) < 20:
                        continue
                    chunks.append({
                        "content": f"{prefix}\n\n{para}",
                        "metadata": {
                            "type": "article",
                            "chapter": current_chapter,
                            "chapter_title": current_chapter_title,
                            "section": current_section,
                            "section_title": current_section_title,
                            "article": art_num,
                            "title": art_title,
                            "paragraph": str(p_idx + 1),
                        },
                    })
            else:
                chunks.append({
                    "content": full_content,
                    "metadata": {
                        "type": "article",
                        "chapter": current_chapter,
                        "chapter_title": current_chapter_title,
                        "section": current_section,
                        "section_title": current_section_title,
                        "article": art_num,
                        "title": art_title,
                        "paragraph": "",
                    },
                })

            i = j
            continue

        i += 1

    return chunks


print("Fonctions de parsing définies ✓")
Fonctions de parsing définies ✓
Cellule 5 — Parsing du AI Act
On exécute le chunker et on affiche les statistiques.

chunks = parse_ai_act()

considerants = [c for c in chunks if c["metadata"]["type"] == "considerant"]
articles     = [c for c in chunks if c["metadata"]["type"] == "article"]

print(f"Nombre total de chunks : {len(chunks)}")
print(f"  - Considérants : {len(considerants)}")
print(f"  - Articles     : {len(articles)}")
print()

# Aperçu de 3 chunks
for c in chunks[:2] + chunks[200:201]:
    print(f"[{c['metadata']['type']}] {c['metadata']['title']}")
    print(c["content"][:150] + "...")
    print()
Nombre total de chunks : 641
  - Considérants : 180
  - Articles     : 461

[considerant] Considérant (1)
Considérant (1) : L’objectif du présent règlement est d’améliorer le fonctionnement du marché intérieur en établissant un cadre juridique uniforme, en...

[considerant] Considérant (2)
Considérant (2) : Le présent règlement devrait être appliqué dans le respect des valeurs de l’Union consacrées dans la Charte, en facilitant la protec...

[article] Pratiques interdites en matière d’IA
Chapitre II - PRATIQUES INTERDITES EN MATIÈRE D’IA > Article 5 : Pratiques interdites en matière d’IA

6.   Les autorités nationales de surveillance d...

Cellule 6 — Construction de l'index FAISS
On encode les 641 chunks avec paraphrase-multilingual-mpnet-base-v2 (278M paramètres, 768 dimensions) et on les indexe dans FAISS.

⏱ Première exécution : ~2-3 min (téléchargement du modèle ~1 Go + encodage).
Exécutions suivantes : ~30s (modèle en cache).

Si l'index existe déjà, cette cellule le recharge simplement.

# Charger le modèle d'embeddings
print(f"Chargement du modèle : {MODEL_NAME}")
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={"device": "cpu"},              # "cuda" si GPU disponible
    encode_kwargs={
        "normalize_embeddings": True,             # Norme L2=1 → cosine similarity = dot product
        "batch_size": 32,                         # Encode 32 textes à la fois
    },
)
print("Modèle chargé ✓")

# Construire ou recharger l'index
if INDEX_DIR.exists():
    print("Index FAISS existant trouvé → rechargement...")
    db = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
else:
    print("Construction de l'index FAISS...")
    documents = [
        Document(page_content=c["content"], metadata=c["metadata"])
        for c in chunks
    ]
    db = FAISS.from_documents(documents, embeddings)
    db.save_local(str(INDEX_DIR))
    print(f"Index sauvegardé dans {INDEX_DIR}/")

print(f"Index FAISS prêt ✓ ({db.index.ntotal} vecteurs de dimension {db.index.d})")
Chargement du modèle : sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5847.42it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Modèle chargé ✓
Index FAISS existant trouvé → rechargement...
Index FAISS prêt ✓ (641 vecteurs de dimension 768)
Cellule 7 — Test de la recherche vectorielle (sans LLM)
Vérifions que FAISS retrouve les bons articles avant de brancher le LLM.

question_test = "Quelles sont les pratiques d'IA interdites ?"

docs = db.similarity_search(question_test, k=TOP_K)

print(f"Question : {question_test}\n")
print(f"{len(docs)} documents retrouvés :\n")
for i, doc in enumerate(docs):
    m = doc.metadata
    label = m.get("title", "")
    if m.get("article"):
        label = f"Article {m['article']} : {label}"
    if m.get("chapter"):
        label = f"Chap. {m['chapter']} > {label}"
    print(f"  {i+1}. [{m['type']}] {label}")
    print(f"     {doc.page_content[:120]}...\n")
Question : Quelles sont les pratiques d'IA interdites ?

5 documents retrouvés :

  1. [article] Chap. II > Article 5 : Pratiques interdites en matière d’IA
     Chapitre II - PRATIQUES INTERDITES EN MATIÈRE D’IA > Article 5 : Pratiques interdites en matière d’IA

1.   Les pratique...

  2. [article] Chap. II > Article 5 : Pratiques interdites en matière d’IA
     Chapitre II - PRATIQUES INTERDITES EN MATIÈRE D’IA > Article 5 : Pratiques interdites en matière d’IA

8.   Le présent a...

  3. [article] Chap. II > Article 5 : Pratiques interdites en matière d’IA
     Chapitre II - PRATIQUES INTERDITES EN MATIÈRE D’IA > Article 5 : Pratiques interdites en matière d’IA

2.   L’utilisatio...

  4. [considerant] Considérant (28)
     Considérant (28) : Si l’IA peut être utilisée à de nombreuses fins positives, elle peut aussi être utilisée à mauvais es...

  5. [considerant] Considérant (132)
     Considérant (132) : Certains systèmes d’IA destinés à interagir avec des personnes physiques ou à générer du contenu peu...

Cellule 8 — Connexion LLM (detection automatique Local/Cloud)
Local : si Ollama repond sur localhost → ChatOllama
Cloud : sinon → ChatHuggingFace (cle HF_TOKEN requise)
import requests as _req

def is_ollama_available():
    try:
        return _req.get("http://localhost:11434/api/tags", timeout=2).status_code == 200
    except Exception:
        return False

if is_ollama_available():
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.1)
    LLM_LABEL = f"Ollama ({OLLAMA_MODEL})"
else:
    from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
    hf_token = os.environ.get("HF_TOKEN", "")
    if not hf_token:
        raise ValueError("Cle HF_TOKEN requise. Definir la variable d'environnement HF_TOKEN.")
    endpoint = HuggingFaceEndpoint(
        repo_id=HF_MODEL,
        huggingfacehub_api_token=hf_token,
        temperature=0.1,
        max_new_tokens=1024,
    )
    llm = ChatHuggingFace(llm=endpoint)
    LLM_LABEL = f"HuggingFace ({HF_MODEL})"

print(f"LLM : {LLM_LABEL}")

# Test rapide
try:
    test = llm.invoke("Reponds OK.")
    content = test.content if hasattr(test, 'content') else str(test)
    print(f"Test : {content[:50]}")
except Exception as e:
    print(f"ERREUR : {e}")
Cellule 9 — Fonctions de base + Retriever + Prompt + Memoire conditionnelle
# ====== Fonctions de base ======

def docstore_lookup(db, filters):
    results = []
    for doc_id in db.index_to_docstore_id.values():
        doc = db.docstore.search(doc_id)
        if all(doc.metadata.get(k) == v for k, v in filters.items()):
            results.append(doc)
    results.sort(key=lambda d: int(d.metadata.get("paragraph", "0") or "0"))
    return results

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

def get_sources(docs):
    sources = []
    for doc in docs:
        m = doc.metadata
        if m.get("type") == "article":
            label = f"Article {m['article']} : {m['title']}"
            if m.get("chapter"):
                label = f"Chapitre {m['chapter']} > {label}"
        else:
            label = m.get("title", "Considerant")
        sources.append(label)
    return sources

# ====== Retriever ======
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": TOP_K, "score_threshold": SCORE_THRESHOLD},
)

# ====== Prompt ======
RESPONSE_PROMPT = """\
Tu es un expert du Reglement europeen sur l'Intelligence Artificielle \
(AI Act, Reglement UE 2024/1689). Reponds en francais.

REGLES :
1. Base ta reponse UNIQUEMENT sur le contexte fourni ci-dessous.
2. Cite les articles et considerants exacts (ex: "Article 6, paragraphe 2").
3. Si le contexte contient des obligations ou interdictions, LISTE-LES.
4. Si le contexte vient d'internet, PRECISE-LE clairement.
5. Ne dis JAMAIS "consultez le texte complet". Utilise ce que tu as.
6. Structure ta reponse avec des titres et des puces si necessaire.
"""

# ====== Memoire ======
chat_history = InMemoryChatMessageHistory()

def is_followup(question):
    q = question.lower().strip()
    explicit = [
        r"\bci.?dessus\b", r"\bprecedent\b", r"\bplus haut\b",
        r"\bta reponse\b", r"\bton analyse\b", r"\bce que tu\b",
        r"\bces articles\b", r"\bces resultats\b", r"\bces points\b",
        r"\bcette liste\b", r"\bce tableau\b", r"\bce texte\b",
    ]
    if any(re.search(p, q) for p in explicit):
        return True
    action_start = [
        r"^(resume|syntheti|explique|detaille|developpe|precise|reformule|tradui)",
        r"^(continue|poursui|complete|approfondi)",
        r"^(et\s+(pour|qu|si|le|la|les|en)\b)",
    ]
    return any(re.search(p, q) for p in action_start)

def call_llm(question, context, context_label="AI Act"):
    messages = [SystemMessage(content=RESPONSE_PROMPT)]
    if is_followup(question):
        for msg in chat_history.messages[-6:]:
            if len(msg.content) > 800:
                messages.append(type(msg)(content=msg.content[:800] + "..."))
            else:
                messages.append(msg)
    messages.append(HumanMessage(
        content=f"Contexte ({context_label}) :\n{context}\n\nQuestion : {question}"
    ))
    return llm.invoke(messages).content

print("Fonctions, retriever, prompt, memoire prets")
Cellule 10 — Outils de recherche
def recherche_article(article_num):
    docs = docstore_lookup(db, {"article": article_num})
    if not docs: return f"Article {article_num} non trouve."
    return format_docs(docs)

def recherche_considerant(numero):
    docs = docstore_lookup(db, {"type": "considerant", "numero": f"({numero})"})
    if not docs: return f"Considerant {numero} non trouve."
    return docs[0].page_content

def recherche_ia_act(query):
    """Retourne (contexte_avec_sources, liste_sources)."""
    docs = retriever.invoke(query)
    if not docs: return "", []
    sources = get_sources(docs)
    header = "Sources AI Act :\n" + "\n".join(f"- {s}" for s in sources) + "\n\nExtraits :\n"
    return header + format_docs(docs), sources

def recherche_web(query):
    search = DuckDuckGoSearchRun()
    parts = []
    try: parts.append(search.invoke(query))
    except Exception: pass
    try: parts.append(search.invoke(query + " results 2025"))
    except Exception: pass
    return "\n\n".join(p for p in parts if p)

print("4 outils prets")
Cellule 11 — Fonction ask() : routage deterministe + memoire
def ask(question):
    print(f"Question : {question}")
    print("=" * 70)
    q = question.lower()

    # === MODE 1 : DIRECT (0 LLM) ===
    art = re.search(r"article\s+(premier|\d+)", q)
    if art:
        num = "1" if art.group(1) == "premier" else art.group(1)
        result = recherche_article(num)
        print(f"[DIRECT] Article {num}\n" + "-" * 70)
        print(result)
        chat_history.add_message(HumanMessage(content=question))
        chat_history.add_message(AIMessage(content=result[:500]))
        return result

    cons = re.search(r"consid[eé]rant\s+(\d+)", q)
    if cons:
        result = recherche_considerant(cons.group(1))
        print(f"[DIRECT] Considerant {cons.group(1)}\n" + "-" * 70)
        print(result)
        chat_history.add_message(HumanMessage(content=question))
        chat_history.add_message(AIMessage(content=result[:500]))
        return result

    # === MODE 2 : RAG (1 LLM) ===
    context, sources = recherche_ia_act(question)
    if context:
        if len(context) > 4000:
            context = context[:4000] + "\n\n[... tronque ...]"
        response = call_llm(question, context, "AI Act")
        print(f"[RAG] {len(sources)} docs : {sources[:4]}\n" + "-" * 70)
        print(response)
        chat_history.add_message(HumanMessage(content=question))
        chat_history.add_message(AIMessage(content=response))
        return response

    # === MODE 3 : WEB (1 LLM) ===
    print("[WEB] Recherche internet...")
    web_results = recherche_web(question)
    if web_results and len(web_results) > 50:
        response = call_llm(question, web_results[:2500], "Internet (DuckDuckGo)")
        print(f"[WEB] DuckDuckGo\n" + "-" * 70)
        print(response)
        chat_history.add_message(HumanMessage(content=question))
        chat_history.add_message(AIMessage(content=response))
        return response

    print("Aucune information trouvee.")
    return "Aucune information trouvee."

print("Fonction ask() prete")
Tests
# Test 1 : DIRECT (article)
ask("Donne-moi l'article 5")
# Test 2 : SUIVI (memoire activee par "resume")
ask("Resume les points cles")
# Test 3 : RAG (question independante, PAS de memoire)
ask("Je recrute par IA, suis-je conforme ?")
Test WEB + memoire
# Test 4 : WEB (question hors AI Act)
ask("Qui a gagne Paris-Roubaix ?")
Verifier la memoire
# Afficher l'historique
print(f"Memoire : {len(chat_history.messages)} messages\n")
for m in chat_history.messages:
    role = "USER" if isinstance(m, HumanMessage) else "AI"
    print(f"  [{role:4s}] {m.content[:80]}")
    print()
Mode interactif
Tapez vos questions en boucle. Entrez quit pour arreter.

print("Agent AI Act — Tapez 'quit' pour quitter")
print("=" * 50)

while True:
    question = input("\nVotre question : ")
    if question.strip().lower() in ("quit", "exit", "q"):
        print("Au revoir !")
        break
    if not question.strip():
        continue
    ask(question)
Agent AI Act — Tapez 'quit' pour quitter
==================================================
Question : date d'entrée en vigueur de l'IA Act?
======================================================================